# DT401 Week 6 Section 2: Data Anonymisation Techniques

**Module:** Data Fundamentals
**Week:** 6 of 7
**Section:** 2 of 3
**Estimated Time:** 60 minutes

---

## Learning Objectives

Understand how to protect personal data using five practical anonymisation techniques available in the `anonympy` library.

By the end of this notebook, you will be able to:
- Apply **pseudonymisation** to replace identifying name fields with synthetic alternatives
- Use **email masking** to protect identifiable contact data whilst preserving analytical value
- Implement **tokenisation** to replace sensitive fields with randomised keys
- Add **numeric perturbation** to obscure precise numerical values whilst preserving statistical properties
- Perform **column suppression** to remove fields that cannot be safely anonymised
- Generate a **summary report** of all anonymisation transformations applied

---

## Context

In Week 6 we are exploring **data anonymisation** — the process of protecting personal data so that individuals cannot be identified, either directly or indirectly. This is a core requirement under UK GDPR (UK Information Commissioner's Office, 2024), and is increasingly expected across data roles in sectors such as healthcare, retail, and local government.

This notebook uses the **`anonympy`** library to demonstrate five practical anonymisation techniques on a fictional customer dataset. You will work through each technique in sequence, applying transformations incrementally and reviewing their effect on the data.

---

## How to Use This Notebook

This notebook contains a mix of instructional text, Python code cells, and interactive exercises. Follow the instructions in each section, run the code cells to see outputs, and complete the exercises to reinforce your learning.

You will find several types of boxes throughout the notebook to guide your learning:

<div class="alert alert-block alert-warning">
<b>Reminder boxes:</b> <br>
These boxes contain important reminders about how this section relates to the overall module and your learning journey.
</div>

<div class="alert alert-block alert-info">
<b>Reflection boxes:</b> <br>
These have pointers as to what we should be looking for or comments on how to approach the topic discussed.
</div>

<div class="alert alert-block alert-success">
<b>Task boxes:</b> <br>
These have questions or tasks you should complete before progressing.

These may include hidden hints or solutions.
<details>
    <summary style="color:black;font-weight:bold">Click here for a hint</summary>

Hints will guide you toward the solution without giving away the answer completely.

</details>

</div>

---

<div class="alert alert-block alert-warning">
<b>Reminder:</b> <br>
As a data professional, you may be asked to share datasets with third parties (e.g. analysts, researchers, or suppliers) whilst ensuring personal data is not exposed. Knowing which anonymisation technique to apply — and why — is an essential skill directly relevant to your DT401 Module Challenge portfolio.
</div>

---

## Part 1: Environment Setup

### Visual Overview

```mermaid
flowchart LR
    A[Raw Customer Data CSV] --> B[pandas DataFrame]
    B --> C[dfAnonymizer Object]
    C --> D[Anonymisation Methods]
    D --> E[Pseudonymisation]
    D --> F[Email Masking]
    D --> G[Tokenisation]
    D --> H[Perturbation]
    D --> I[Suppression]
    E & F & G & H & I --> J[Anonymised Output]
```

**What this diagram shows:**
The dependency chain from raw customer data through the `dfAnonymizer` wrapper to the five anonymisation techniques and their combined anonymised output.

**Why this matters in your role:**
Understanding the library architecture helps you predict the correct function to use for each column type, and prevents errors caused by applying mismatched techniques.

---

### Understanding the `anonympy` Library

The `anonympy` library provides a `dfAnonymizer` class that wraps a Pandas DataFrame and exposes anonymisation methods grouped by data type:

- **Categorical methods** — for text fields (names, emails, addresses)
- **Numeric methods** — for integer and float fields (age, salary)
- **Datetime methods** — for date and timestamp fields

Before applying any technique, `dfAnonymizer` inspects your DataFrame and classifies columns automatically. This classification drives which methods are valid for each column. Applying a categorical method to a numeric column (or vice versa) will raise an error — so checking column types first is always good practice (Melis et al., 2019).

<div class="alert alert-block alert-warning">
<b>Reminder:</b> <br>
If you have not yet installed <code>anonympy</code>, uncomment and run the cell below, then restart the kernel before continuing. You only need to do this once per environment.
</div>

In [ ]:
# Uncomment the lines below ONLY if anonympy is not yet installed
# !test -d open-data-anonimizer || git clone --depth 1 https://github.com/ArtLabss/open-data-anonimizer.git
# %pip install -e ./open-data-anonimizer
# !sudo apt-get update 2>&1 | grep -v "yarn" || true
# !sudo apt-get install -y --no-install-recommends libgl1

**Worked Example — Importing libraries:**

In [ ]:
import pandas as pd                       # DataFrame operations
from anonympy.pandas import dfAnonymizer  # Anonymisation wrapper class

<div class="alert alert-block alert-info">
<b>What just happened?</b><br>
<ul>
<li><code>pandas</code> is imported for data loading and display.</li>
<li><code>dfAnonymizer</code> is the core class from <code>anonympy</code> — it wraps a Pandas DataFrame and provides all anonymisation methods grouped by column type.</li>
</ul>
</div>

---

## Part 2: Loading and Inspecting Customer Data

### Visual Overview

```mermaid
flowchart LR
    A[customer_data.csv] --> B[pd.read_csv]
    B --> C[DataFrame]
    C --> D[dfAnonymizer]
    D --> E[numeric_columns<br/>age]
    D --> F["categorical_columns<br/>first_name, last_name<br/>email_address<br/>street_address, post_code"]
    D --> G[datetime_columns<br/>date_of_birth]
```

**What this diagram shows:**
The column types in our customer dataset and how `dfAnonymizer` classifies them into numeric, categorical, and datetime groups.

**Why this matters:**
Choosing the correct anonymisation technique depends entirely on the column type. Matching technique to type is both a technical requirement (wrong types raise errors) and a governance requirement (appropriate techniques depend on the nature of the data).

---

### Understanding Column Type Classification

Our fictional dataset represents customer records from a retail loyalty scheme. It contains a mix of personal identifiers (names, email), contact data (address, postcode), a numeric field (age), and a date field (date of birth).

When we pass this DataFrame to `dfAnonymizer`, it automatically inspects each column's dtype and categorises them. Printing these categories before applying any technique is best practice — it prevents mismatches and helps you plan your anonymisation strategy systematically (UK ICO, 2023).

<div class="alert alert-block alert-warning">
<b>Reminder:</b> <br>
The <code>parse_dates</code> parameter tells Pandas to interpret the <code>date_of_birth</code> column as a datetime object, not a plain string. Without this, datetime anonymisation methods will not work correctly.
</div>

**Worked Example — Loading data and classifying columns:**

In [ ]:
# Load fictional customer data; parse date_of_birth as datetime
customer_data = pd.read_csv(
    'customer_data.csv',
    parse_dates=['date_of_birth'],
    dayfirst=True
)
customer_data.head()

In [ ]:
# Create anonympy object and inspect column classifications
anon_data = dfAnonymizer(customer_data)
print('Numeric columns:', anon_data.numeric_columns)
print('Categorical columns:', anon_data.categorical_columns)
print('Date columns:', anon_data.datetime_columns)

<div class="alert alert-block alert-info">
<b>What just happened?</b><br>
<ul>
<li><code>pd.read_csv()</code> loaded the file into a DataFrame; <code>parse_dates</code> ensured <code>date_of_birth</code> is treated as a datetime.</li>
<li><code>dfAnonymizer(customer_data)</code> created the anonympy wrapper object <code>anon_data</code>.</li>
<li>Printing <code>.numeric_columns</code>, <code>.categorical_columns</code>, and <code>.datetime_columns</code> revealed the three column groups — this tells us exactly which methods apply to which fields.</li>
</ul>
</div>

<div class="alert alert-block alert-success">
<b>Task 2.1 — Explore the data:</b> <br>
Before continuing, answer these questions by writing and running code below:

1. How many rows are in the dataset? (hint: <code>len()</code> or <code>.shape</code>)
2. What is the dtype of the <code>date_of_birth</code> column? (hint: <code>.dtypes</code>)

<details>
    <summary style="color:black;font-weight:bold">Click here for a hint</summary>

```python
print(len(customer_data))
print(customer_data.dtypes)
```

</details>

</div>

In [ ]:
# Task 2.1 - How many rows? What is the dtype of date_of_birth?


---

## Part 3: Pseudonymisation

### Visual Overview

```mermaid
flowchart LR
    A["first_name: Emily, James, Sarah"] --> B[categorical_fake]
    B --> C["first_name: Laura, Connor, Aisha"]
    D["last_name: Clarke, Thornton, Patel"] --> E[categorical_fake]
    E --> F["last_name: Bennett, Walsh, Singh"]
```

**What this diagram shows:**
Real names replaced with synthetic (fake) names using a locale-aware name generator. The structure of the data is preserved but no individual is identifiable.

**Why this matters:**
Pseudonymisation is the most common first step in anonymising personal data — it removes the most obvious identifiers whilst preserving data structure for downstream processing.

---

### Understanding Pseudonymisation

**Pseudonymisation** replaces directly identifying fields (such as names) with realistic but fictional alternatives. Under UK GDPR, pseudonymised data is still considered personal data — because re-identification may be possible if the substitution key is available — but pseudonymisation is recognised as a valuable risk-reduction measure (UK ICO, 2023).

The `anonympy` method `categorical_fake()` uses the `Faker` library under the hood to generate locale-appropriate synthetic names. This means the output looks realistic and does not disrupt downstream processing that expects name-formatted strings.

<div class="alert alert-block alert-warning">
<b>Reminder:</b> <br>
<code>categorical_fake()</code> generates new random values each time it is called. If you re-run the cell, the synthetic names will change — this is expected behaviour. The original values in <code>customer_data</code> are not affected; only the <code>anon_data</code> object is modified.
</div>

**Worked Example — Pseudonymising first and last names:**

In [ ]:
anon_data.categorical_fake('first_name')  # Replace first_name with synthetic names
anon_data.categorical_fake('last_name')   # Replace last_name with synthetic names
anon_data.to_df()                         # Display updated DataFrame

<div class="alert alert-block alert-info">
<b>What just happened?</b><br>
<ul>
<li><code>categorical_fake('first_name')</code> replaced each real first name with a randomly generated synthetic name.</li>
<li><code>categorical_fake('last_name')</code> did the same for surnames.</li>
<li><code>to_df()</code> returned the DataFrame with all transformations applied so far.</li>
<li>The original names are now removed from the working data — but note that <code>customer_data</code> still holds the original values.</li>
</ul>
</div>

**Common Mistake — Applying categorical methods to numeric columns:**

In [ ]:
# This will raise an error — categorical_fake() cannot be applied to numeric columns
# anon_data.categorical_fake('age')  # age is numeric, not categorical

<div class="alert alert-block alert-success">
<b>Task 3.1 — Pseudonymise a phone number field:</b> <br>
Imagine the dataset also contains a <code>phone_number</code> column. Using <code>categorical_fake()</code>, write code to pseudonymise it. Then call <code>to_df()</code> to check the result.

<details>
    <summary style="color:black;font-weight:bold">Click here for a hint</summary>

Use <code>anon_data.categorical_fake('phone_number')</code> — same pattern as the name columns.

</details>

<details>
    <summary style="color:green;font-weight:bold">Click here for the solution</summary>

```python
anon_data.categorical_fake('phone_number')
anon_data.to_df()
```

</details>

</div>

In [ ]:
# Task 3.1 - Pseudonymise phone_number if it exists in your dataset


<div class="alert alert-block alert-info">
<b>Success criteria:</b><br>
<ul>
<li><code>first_name</code> and <code>last_name</code> columns contain synthetic, non-real names.</li>
<li><code>to_df()</code> displays the updated DataFrame without errors.</li>
<li>The original <code>customer_data</code> DataFrame is unchanged.</li>
</ul>
</div>

---

## Part 4: Email Masking

### Visual Overview

```mermaid
flowchart LR
    A["emily.clarke@gmail.com"] --> B[categorical_email_masking]
    B --> C["***@gmail.com"]
    D["james.thornton@nhs.net"] --> E[categorical_email_masking]
    E --> F["***@nhs.net"]
```

**What this diagram shows:**
Email addresses before and after masking. The domain is preserved; the local part (before `@`) is obscured with asterisks.

**Why this matters:**
Partial masking balances data utility with privacy. Domain metadata (e.g. `@nhs.net`) may still be analytically useful for segmenting by organisation type, but the individual is no longer identifiable from the address alone.

---

### Understanding Email Masking

**Masking** replaces sensitive characters with placeholder symbols (typically asterisks) while preserving the structure and partial content of the original value. For email addresses, retaining the domain is often valuable — an analyst might legitimately need to know that a customer uses an NHS email, for example, without needing the specific individual's address.

The `categorical_email_masking()` method from `anonympy` preserves the domain (everything after `@`) while masking the local part. This is a common approach in healthcare and finance contexts where domain-level segmentation is analytically valuable (ENISA, 2023).

<div class="alert alert-block alert-warning">
<b>Reminder:</b> <br>
Email masking is a one-way operation in this workflow — once applied, the original email addresses cannot be recovered from the <code>anon_data</code> object. Ensure you have a separate copy of the originals (such as the <code>customer_data</code> DataFrame) if re-identification is ever needed under authorised conditions.
</div>

**Worked Example — Masking email addresses:**

In [ ]:
anon_data.categorical_email_masking('email_address')  # Mask email local parts
anon_data.to_df()                                     # Display result

<div class="alert alert-block alert-info">
<b>What just happened?</b><br>
<ul>
<li><code>categorical_email_masking()</code> replaced the local part of each email (before <code>@</code>) with asterisks.</li>
<li>The domain (e.g. <code>@gmail.com</code>, <code>@nhs.net</code>) was preserved in full.</li>
<li>This allows domain-level analysis (e.g. "how many customers use NHS email?") without exposing individual identities.</li>
</ul>
</div>

<div class="alert alert-block alert-success">
<b>Task 4.1 — Reflect on masking trade-offs:</b> <br>
In the markdown cell below, write 2–3 sentences answering:

When might preserving the email domain be a <b>privacy risk</b> rather than a benefit?

<details>
    <summary style="color:black;font-weight:bold">Click here for a hint</summary>

Think about niche or identifiable domains — for example, a small community organisation or a specific government department where only a handful of people might have email accounts.

</details>

</div>

*Your answer here...*

---

## Part 5: Tokenisation

### Visual Overview

```mermaid
flowchart LR
    A["street_address: 14 Maple Street"] --> B[categorical_tokenization]
    B --> C["street_address: a7f3b2c9"]
    D["post_code: SW1A 1AA"] --> E[categorical_tokenization]
    E --> F["post_code: d4e1f8a2"]
```

**What this diagram shows:**
Original address and postcode values replaced with random alphanumeric tokens. There is no structural or mathematical relationship between the token and the original value.

**Why this matters:**
Tokenisation is stronger than masking for high-risk fields — the token cannot be reverse-engineered without access to a separately stored mapping table. This makes it appropriate for fields like postcodes where even partial masking would still expose geographic information.

---

### Understanding Tokenisation

**Tokenisation** replaces sensitive data values with randomly generated tokens — alphanumeric strings that have no mathematical or structural relationship to the original values. Unlike encryption, tokens cannot be reverse-engineered algorithmically; re-identification requires access to a separately stored mapping table.

This makes tokenisation particularly appropriate for fields like street addresses and postcodes, where partial masking would still expose geographic information. In the financial sector, tokenisation is routinely used for payment card numbers (PCI DSS compliance); in healthcare, it is applied to patient record identifiers (NHS Digital, 2024).

<div class="alert alert-block alert-warning">
<b>Reminder:</b> <br>
Tokens are not deterministic — running tokenisation twice on the same value will produce different tokens each time. If you need consistent tokens across runs (e.g. for referential integrity between tables), you must store and re-use the mapping table from the first run.
</div>

**Worked Example — Tokenising address and postcode:**

In [ ]:
anon_data.categorical_tokenization('street_address')  # Tokenise street address
anon_data.categorical_tokenization('post_code')       # Tokenise postcode
anon_data.to_df()                                     # Display result

<div class="alert alert-block alert-info">
<b>What just happened?</b><br>
<ul>
<li><code>categorical_tokenization()</code> replaced each unique address and postcode value with a randomly generated alphanumeric token.</li>
<li>The tokens cannot be decoded without the original mapping — they carry no geographic meaning.</li>
<li>Row-level joins are still possible using tokens (e.g. if two records shared the same address, they now share the same token).</li>
</ul>
</div>

<div class="alert alert-block alert-success">
<b>Task 5.1 — Tokenise an additional field:</b> <br>
Apply <code>categorical_tokenization()</code> to a field called <code>national_insurance_number</code> (if present in your dataset). Then call <code>to_df()</code>.

<details>
    <summary style="color:green;font-weight:bold">Click here for the solution</summary>

```python
anon_data.categorical_tokenization('national_insurance_number')
anon_data.to_df()
```

</details>

</div>

In [ ]:
# Task 5.1 - Tokenise national_insurance_number


<div class="alert alert-block alert-info">
<b>Success criteria:</b><br>
<ul>
<li><code>street_address</code> and <code>post_code</code> contain alphanumeric tokens, not real values.</li>
<li>Tokens contain no recognisable structure from the originals.</li>
<li><code>to_df()</code> displays without errors.</li>
</ul>
</div>

---

## Part 6: Numeric Perturbation and Column Suppression

### Visual Overview

```mermaid
flowchart LR
    A[age: 34, 47, 29] --> B[numeric_noise]
    B --> C["age: 35.2, 46.1, 28.7<br/>(perturbed)"]
    D[date_of_birth column] --> E[column_suppression]
    E --> F[column removed from output]
    C --> G[Anonymised Dataset]
    F --> G
```

**What this diagram shows:**
Age values before and after noise addition; the `date_of_birth` column removed entirely via suppression. Both operations are needed together because the two fields are correlated.

**Why this matters:**
Numeric fields require different anonymisation strategies to categorical fields. Sometimes adding noise is sufficient; other times, full suppression is the only safe option.

---

### Understanding Perturbation and Suppression

**Perturbation** (or numeric noise) adds small random values to numerical fields, making precise re-identification impossible whilst preserving statistical properties such as mean and distribution shape. This is widely used in healthcare research datasets where aggregate analysis must remain valid (El Emam et al., 2020).

However, perturbation alone is not always sufficient. In our dataset, `age` and `date_of_birth` are correlated — an attacker could recalculate true age from the date of birth. **Column suppression** removes a field entirely from the output, and is the appropriate technique when a field creates re-identification risk that cannot be mitigated by transformation alone.

<div class="alert alert-block alert-warning">
<b>Reminder:</b> <br>
When applying perturbation to <code>age</code>, consider whether <code>date_of_birth</code> remains in the dataset. If it does, the perturbed age can be checked against the birth date, potentially revealing the original value. Apply suppression to <code>date_of_birth</code> to prevent this.
</div>

**Worked Example — Adding numeric noise to age:**

In [ ]:
anon_data.numeric_noise('age')  # Add random noise to age values
anon_data.to_df()               # Display result

<div class="alert alert-block alert-info">
<b>What just happened?</b><br>
<ul>
<li><code>numeric_noise()</code> added small random values to each entry in the <code>age</code> column.</li>
<li>The adjusted values are close to the originals but not identical — aggregate statistics (mean, median) remain approximately valid.</li>
<li>However, <code>date_of_birth</code> still exists in the dataset and could be used to recalculate true age.</li>
</ul>
</div>

**Worked Example — Suppressing the date_of_birth column:**

In [ ]:
anon_data.column_suppression('date_of_birth')  # Remove date_of_birth entirely
anon_data.to_df()                              # Display result

<div class="alert alert-block alert-info">
<b>What just happened?</b><br>
<ul>
<li><code>column_suppression()</code> removed <code>date_of_birth</code> from the output DataFrame entirely.</li>
<li>This eliminates the risk that an attacker could verify or recalculate the perturbed <code>age</code> values.</li>
<li>Suppression is the strongest anonymisation technique — it removes data rather than transforming it.</li>
</ul>
</div>

<div class="alert alert-block alert-success">
<b>Task 6.1 — Independent task: Anonymise a salary dataset:</b> <br>
Imagine you have a DataFrame with columns <code>employee_id</code>, <code>salary</code>, and <code>start_date</code>. Decide which anonymisation technique to apply to each column, then write the code.

<details>
    <summary style="color:black;font-weight:bold">Click here for a hint</summary>

Consider the nature of each field:
<ul>
<li><code>employee_id</code> — unique identifier: consider <b>tokenisation</b></li>
<li><code>salary</code> — numeric: consider <b>perturbation</b></li>
<li><code>start_date</code> — datetime that may reveal tenure: consider <b>suppression</b> or datetime perturbation</li>
</ul>

</details>

<details>
    <summary style="color:green;font-weight:bold">Click here for the solution</summary>

```python
# Assumes anon_salary = dfAnonymizer(salary_df)
anon_salary.categorical_tokenization('employee_id')
anon_salary.numeric_noise('salary')
anon_salary.column_suppression('start_date')
anon_salary.to_df()
```

</details>

</div>

In [ ]:
# Task 6.1 - Design and write an anonymisation pipeline for a salary dataset
# Columns: employee_id (categorical), salary (numeric), start_date (datetime)


<div class="alert alert-block alert-info">
<b>Success criteria:</b><br>
<ul>
<li><code>age</code> values have been perturbed (slightly different from originals).</li>
<li><code>date_of_birth</code> column is no longer present in the output.</li>
<li>You can explain why suppression was chosen for <code>date_of_birth</code>.</li>
<li>Independent task applies at least two different anonymisation techniques.</li>
</ul>
</div>

---

## Part 7: Anonymisation Summary Report

### Visual Overview

```mermaid
flowchart LR
    A[anon_data.info] --> B[Summary Output]
    B --> C["first_name → synthetic data"]
    B --> D["last_name → synthetic data"]
    B --> E["email_address → partial masking"]
    B --> F["street_address → tokenisation"]
    B --> G["post_code → tokenisation"]
    B --> H["age → numeric perturbation"]
    B --> I["date_of_birth → suppressed"]
```

**What this diagram shows:**
The `.info()` output structure, mapping each column in the dataset to its applied anonymisation technique.

**Why this matters:**
Documenting which techniques were applied to which columns is a compliance requirement — it forms part of a data processing record under UK GDPR Article 30.

---

### Understanding Anonymisation Audit Trails

Data protection regulations require organisations to maintain records of their data processing activities. For anonymisation workflows, this means documenting which technique was applied to each field, by whom, and when. The `anonympy` `.info()` method provides a programmatic summary that can serve as the starting point for such a record (UK ICO, 2024).

<div class="alert alert-block alert-warning">
<b>Reminder:</b> <br>
The <code>.info()</code> summary describes the transformations applied within this session only. For a full audit trail, you would typically log this output alongside the run timestamp, operator name, and data source reference to satisfy the record-keeping requirements of UK GDPR Article 30.
</div>

**Worked Example — Generating the anonymisation summary:**

In [ ]:
anon_data.info()  # Display summary of all anonymisation techniques applied

<div class="alert alert-block alert-info">
<b>What just happened?</b><br>
<ul>
<li><code>.info()</code> returned a structured summary showing each column and the anonymisation method applied.</li>
<li>This confirms: <code>first_name</code> and <code>last_name</code> → synthetic data; <code>date_of_birth</code> → suppressed; <code>age</code> → numeric perturbation; <code>email_address</code> → partial masking; <code>street_address</code> and <code>post_code</code> → tokenisation.</li>
<li>This output is suitable as a starting record for a GDPR Article 30 processing log.</li>
</ul>
</div>

---

## Reflection

Reflect on the following two questions and record your thoughts in the cell below. Ground your answers in examples from your workplace, using **approximately 300 words** in total.

Your answers here form part of your Module Challenge Part A portfolio and are assessed against:
- **MLO1:** *Explain the roles and functions of technology solutions within an organisation*
- **MLO2:** *Critically reflect upon data management systems and their role in digital and technology solutions*

Where you draw on external sources, cite them in Harvard style. This does not count towards your word count.

Once you commit your notebook, your Coach will be able to provide feedback on your responses. You can develop your answers further before your final submission.

<div class="alert alert-block alert-info">

1. Where would a tool like `anonympy` sit in your organisation's data pipeline? Which roles would typically configure, run or review it, and why? (MLO1)

2. In this notebook before choosing an anonymisation technique, we classified the columns as numeric, categorical or datetime, similar to the data type classification you carried out in Week 2. Is this classification step ever safe to skip, or does it always need to happen first? Justify your view with reference to anonymisation, validation, or another data management action. (MLO2)

</div>

*Write your responses here.*

1. 

2. 

References (if applicable):

---

## Summary

**You have completed Week 6 Section 2.** You can now:

- Set up the anonymisation environment by importing `pandas` and the `dfAnonymizer` class from `anonympy`
- Load customer data with `pd.read_csv()` and inspect column classifications using `anon_data.numeric_columns`, `.categorical_columns`, and `.datetime_columns`
- Apply pseudonymisation to name fields using `categorical_fake()` to generate realistic synthetic alternatives
- Mask email addresses using `categorical_email_masking()` to preserve domain while protecting individual identity
- Tokenise high-risk fields using `categorical_tokenization()` to replace values with non-reversible random keys
- Add numeric noise using `numeric_noise()` and suppress the `date_of_birth` column with `column_suppression()` to prevent age recalculation
- Generate an anonymisation summary report using `.info()` suitable for a GDPR Article 30 processing record

**Key takeaways:**

- Reversibility varies by technique: tokenisation can be reversed only with a separately stored mapping table, while masking and suppression permanently remove the ability to recover the original values.
- `dfAnonymizer` classifies columns as numeric, categorical, or datetime, and each method only works on its matching column type; applying a technique to the wrong column type raises an error.

**What comes next:**

- **Section 3** extends these anonymisation techniques to geospatial data, applying coordinate rounding and perturbation to protect location data that can identify individuals even without other personal identifiers.

**Action items:**

1. Save this notebook (Ctrl+S or Cmd+S)
2. Commit and push your work to your GitHub repository with the commit message `"Complete Week 6 Section 2 data anonymisation techniques"`
3. Make sure the repository shows your latest commit and that all files are up to date
4. Share a link to your repository with your Coach in the DT401.6 component on Aptem
5. Open `DT401_Week6_Section3.ipynb` when ready to continue


---

## References

El Emam, K., Jonker, E. and Arbuckle, L. (2020) *Anonymizing Health Data: Case Studies and Methods to Get You Started*. Sebastopol, CA: O'Reilly Media.

ENISA (2023) *Pseudonymisation Techniques and Best Practices*. Heraklion: European Union Agency for Cybersecurity. Available at: https://www.enisa.europa.eu/publications/pseudonymisation-techniques-and-best-practices (Accessed: 1 March 2026).

Melis, M., Demontis, A., Biggio, B., Fumera, G. and Roli, F. (2019) 'How deep learning sees the world: a survey on adversarial attacks and defenses', *arXiv preprint*, arXiv:1804.00097.

NHS Digital (2024) *Data Security Standards: Anonymisation and Pseudonymisation*. Leeds: NHS Digital. Available at: https://digital.nhs.uk/data-and-information/looking-after-information/data-security-and-information-governance (Accessed: 1 March 2026).

UK ICO (2023) *Anonymisation, Pseudonymisation and Privacy Enhancing Technologies Guidance*. Wilmslow: Information Commissioner's Office. Available at: https://ico.org.uk/media/about-the-ico/consultations/4019579/chapter-2-anonymisation-draft-guidance.pdf (Accessed: 1 March 2026).

UK ICO (2024) *Guide to the UK General Data Protection Regulation (UK GDPR)*. Wilmslow: Information Commissioner's Office. Available at: https://ico.org.uk/for-organisations/uk-gdpr-guidance-and-resources/ (Accessed: 1 March 2026).